<a href="https://colab.research.google.com/github/snr-laboratory/snrlab-ic-q-pix-v1/blob/chip_network_sim/chip_network_sim/dev_journals/kgosine_202603_architecture.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## Purpose


Simulate an m x n digital chip network where each chip is described with RTL. Each chip rceives data from one upstream neighbor and sends data to one downstream neighbor. Each chip also generates data locally. Data traffic is buffered using a single local FIFO (defined via RTL). The routing of data through the network is defined via the Orchestrator.

## Chip Unit

A chip functions as a 2-input, 1-ouput unit. The first input is data from one upstream chip. The second input is data generated locally on the chip. The one output is passed to the downstream chip.

The FIFO on the chip is defined with RTL (2-input, 1-output) which buffers data traffice through the chip.

The locally generated data can be viewed as any analog or mixed-signal front-end structure within the chip which generates data-packets. In this example, this front-end is generalized to a random number generator

Generated packets:
lines [666:680] chip_rtl.cpp
A packet is generated when a RNG selects a draw < gen_ppm where gen_ppm is an int between 0-1,000,000 that is passed as an argument to each chip. gen_ppm acts as a probability (rate) at which packets are generated.

The contents of a data packet are the following 64-bits:
src_id: 16 bits for chip id
timestamp: 24 bits for timestamp
payload: 24 bits for data payload

The payload is a randomly generated 24 bits. The timestamp is incrememented with each clock pulse.

## Config

run_from_config.py is an orchestrator launcher which passes arguments defined in a config.json to the Orchestrator (orchestrator.c). See network_3x4_snake_to_top_left.json for an example of how the routing, data packet, fifo_depth, and data packet generation can be set.


## Orchestrator

orchestrator.c is the top-level simualtion driver. it parses run settings (which can be passed using config.json) and then launches one chip process per grid cell. It is responsible for running the lock-step control protocol; it passes a TICK to all chips and waits to collect DONE from all chips, enforcing that all chips are stepping together in time. The orchestrator will use nng sockets for orchestrator-to-chp control/metrics to confirm time-stepping.



## Future Directions


* incorporating a Spice (analog) portion into the simulation

we've previously successfully performed Spice/RTL cosimulation using Verilator before

* allowing data input from any of the four adjacent chips

this will allow for configuration routes beyond daisy chaining which is obviously vulnerable to single point failure. at this stage, the routing would still be defined at the top level.

* incorpoarting existing RTL (LArPix) including more realistic data packet creation and transmission timing.



## Applications

* because of the distributed network design of the architecture, chips on the order of those needed in DUNE: for 4x4mm pixels, with 62,500 pixels per m^2. Assuming 64 channel chips, on the order of 10^3 chips/ m^2. Assuming pixelating just one face of a DUNE module, that's 380 m^2 so somewhere around 380,000 chips. the only way to do a simualtion of this scale is with distributed networking.

* taking a design from RTL/Spice (transistor-level) to a tile-level system reduces the number of edge cases which are missed. No part of the simualtion path from chip to tile is simplified, increasing confidence of final product. any unintended effects from the transistor-level or communication design wil appear in the simulation results.

* modifiable to any generic chip: highlighted with this toy model (and later by incorporating analog components), this architecture allows for plugging in any generic chip design or chip routing scheme. Further, the communication system can be tweaked for ACK/NACK signals to be included allowing for testing of new networking schemes with minimal changes to the software. The chip design needs only to be compiled with Verilator once and then can quickly be used in a tile simualtion.

* simple simualtions can show packet loss as a function of FIFO sizes, maximum data generation rates, routing schemes ect.

